In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#Read in file
df = pd.read_csv(
    "https://raw.githubusercontent.com/GloBIAS-BioimageAnalysts/Survey_2024/main/data/survey2024_headersandcommascleaned_recleaned.csv"
)

In [4]:
#Clean data
df.columns = df.columns.str.strip().str.replace('\n', ' ', regex=True)

In [5]:
df['What is your location'] = df['What is your location'].replace({
    'Australia/ Oceania': 'Australia/Oceania',
    'Near/ Middle East': 'Middle East'
})

In [6]:
#Summary of counts for each region
region_counts = df['What is your location'].value_counts().reset_index()
region_counts.columns = ['Region', 'Count']
region_counts

,Region,Count
0,Europe,178
1,North America,46
2,South America,28
3,Australia/Oceania,12
4,Asia,10
5,Middle East,8
6,Africa,6
7,Central America,1


In [7]:
df = df.rename(columns={
    'Which of these describe your position Choose all options that apply': 'PositionMulti'
})

In [8]:
df['PositionMulti'] = df['PositionMulti'].fillna('')

df['PositionMulti'] = df['PositionMulti'].apply(
    lambda x: [i.strip() for i in x.split(',') if i.strip() != '']
)

In [9]:
region_col = 'What is your location'

region_counts = (
    df[region_col]
    .value_counts(dropna=False)
    .rename_axis('Region')
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)

region_counts['Percent'] = 100 * region_counts['Count'] / region_counts['Count'].sum()
region_counts

,Region,Count,Percent
0,Europe,178,61.379310
1,North America,46,15.862069
2,South America,28,9.655172
3,Australia/Oceania,12,4.137931
4,Asia,10,3.448276
5,Middle East,8,2.758621
6,Africa,6,2.068966
7,Central America,1,0.344828
8,NaN,1,0.344828


In [10]:
region_position_counts = (
    df.assign(PositionMulti=df['PositionMulti'])  # use as-is
      .explode('PositionMulti')
      .dropna(subset=['PositionMulti'])
      .query("PositionMulti != ''")
      .groupby([region_col, 'PositionMulti'])
      .size()
      .rename('Count')
      .reset_index()
      .sort_values(['Count'], ascending=False)
)

region_position_counts.head()

,What is your location,PositionMulti,Count
37,Europe,staff scientist in a core facility,63
31,Europe,core facility head,35
33,Europe,postdoctoral researcher,32
28,Europe,PhD student,28
38,Europe,staff scientist researcher,23


In [11]:
region_position_percent = (
    region_position_counts
    .join(
        region_position_counts.groupby(region_col)['Count'].sum().rename('RegionTotal'),
        on=region_col
    )
)
region_position_percent['Percent'] = 100 * region_position_percent['Count'] / region_position_percent['RegionTotal']
region_position_percent = region_position_percent.drop(columns='RegionTotal')
region_position_percent.head()

,What is your location,PositionMulti,Count,Percent
37,Europe,staff scientist in a core facility,63,30.000000
31,Europe,core facility head,35,16.666667
33,Europe,postdoctoral researcher,32,15.238095
28,Europe,PhD student,28,13.333333
38,Europe,staff scientist researcher,23,10.952381


In [15]:
region_col = 'What is your location'

# Explode and count Region × Position
rp = (
    df.explode('PositionMulti')
      .dropna(subset=['PositionMulti', region_col])
      .query("PositionMulti != ''")
      .groupby([region_col, 'PositionMulti'])
      .size()
      .rename('Count')
      .reset_index()
)

# Region totals (total selections per region)
region_totals = rp.groupby(region_col)['Count'].sum().rename('RegionTotal')

# Pick the top Position per Region (single winner)
idx = rp.groupby(region_col)['Count'].idxmax()
top1_per_region = rp.loc[idx].rename(columns={'PositionMulti': 'TopPosition'})

# Merge totals and compute percent
top1_per_region = (
    top1_per_region
    .merge(region_totals, on=region_col, how='left')
    .assign(Percent=lambda d: 100 * d['Count'] / d['RegionTotal'])
    .sort_values(region_col)
    .reset_index(drop=True)
)

top1_per_region

,What is your location,TopPosition,Count,RegionTotal,Percent
0,Africa,postdoctoral researcher,4,9,44.444444
1,Asia,PhD student,2,11,18.181818
2,Australia/Oceania,staff scientist in a core facility,6,14,42.857143
3,Central America,professor,1,1,100.000000
4,Europe,staff scientist in a core facility,63,210,30.000000
5,Middle East,staff scientist in a core facility,6,9,66.666667
6,North America,staff scientist in a core facility,15,56,26.785714
7,South America,PhD student,10,31,32.258065


In [22]:
sector_col = 'In which of these sectors do you work'

df_sector_long = (
    df.explode(sector_col)
      .dropna(subset=[sector_col])
      .query(f"`{sector_col}` != ''")
)

sector_counts = (
    df_sector_long[sector_col]
    .value_counts()
    .rename_axis('Sector')
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)

sector_counts

,Sector,Count
0,university,173
1,research institution,147
2,company,15
3,self employed,1


In [25]:
# Ensure a respondent ID
df_with_id = df.reset_index().rename(columns={'index': 'RespondentID'})

df_sector_long_id = (
    df_with_id.explode(sector_col)
              .dropna(subset=[sector_col, region_col])
              .query(f"`{sector_col}` != ''")
)

# Count unique respondents per Region × Sector
sector_by_region_resp = (
    df_sector_long_id
    .groupby([region_col, sector_col])['RespondentID']
    .nunique()
    .rename('Count')
    .reset_index()
)

# Unique respondents per region (denominator)
region_totals_resp = (
    df_sector_long_id.groupby(region_col)['RespondentID'].nunique().rename('RegionTotal')
)

# Top sector per region by unique respondents
idx = sector_by_region_resp.groupby(region_col)['Count'].idxmax()

top_sector_per_region_resp = (
    sector_by_region_resp.loc[idx]
        .rename(columns={sector_col: 'TopSector'})
        .merge(region_totals_resp, on=region_col, how='left')
        .assign(Percent=lambda d: 100 * d['Count'] / d['RegionTotal'])
        .sort_values(region_col)
        .reset_index(drop=True)
)

top_sector_per_region_resp

,What is your location,TopSector,Count,RegionTotal,Percent
0,Africa,university,6,6,100.000000
1,Asia,research institution,6,10,60.000000
2,Australia/Oceania,university,8,12,66.666667
3,Central America,research institution,1,1,100.000000
4,Europe,university,114,178,64.044944
5,Middle East,university,6,8,75.000000
6,North America,university,22,46,47.826087
7,South America,research institution,22,28,78.571429
